# 01: Pitcher arsenal

One pitcher-season from the warehouse: data check, arsenal (pandas and SQL), outcomes, usage by handedness, and a movement chart.

**Before running:** pull the pitcher into the warehouse from Terminal (only needed once per pitcher-season):
```
uv run bb pull skubal tarik 2025
```
Then set the pitcher below and click **Run All**. Set `USE_SAMPLE = True` to run on fictional data with no internet.

In [ ]:
PITCHER = ("skubal", "tarik")   # (last, first)
SEASON = 2025
PHASE = "regular"                # "regular", "postseason", "spring" or "all"
USE_SAMPLE = False               # True = fictional data
MIN_N = 20                       # pitch types thrown fewer times than this are left out

In [ ]:
import pandas as pd

from baseball import warehouse
from baseball.evaluation import arsenal, arsenal_sql, outcomes, usage_by_hand
from baseball.evaluation.arsenal import ARSENAL_SQL
from baseball.ingest.sample import fictional_pitcher
from baseball.phases import phase
from baseball.viz import movement_plot

pd.set_option("display.max_columns", 50)

## 1. Load from the warehouse

In [ ]:
if USE_SAMPLE:
    raw = fictional_pitcher(season=SEASON)
    label = "Fictional pitcher (sample data)"
else:
    con = warehouse.connect()
    pid = warehouse.find_pitcher(con, *PITCHER)
    raw = warehouse.pitcher_pitches(con, pid, SEASON) if pid else pd.DataFrame()
    con.close()  # release the file so `bb pull` can write to it from Terminal
    if raw.empty:
        raise SystemExit(f"Not in the warehouse yet. In Terminal: uv run bb pull {PITCHER[0]} {PITCHER[1]} {SEASON}")
    label = f"{PITCHER[1].title()} {PITCHER[0].title()}, {SEASON} {PHASE}"

print("All pitches by game type:", raw["game_type"].value_counts().to_dict())
pitches = phase(raw, PHASE)
print(f"{label}: {len(pitches):,} pitches in {pitches['game_pk'].nunique()} games")

## 2. Check the data before trusting it
Missing values in the columns the tables use, and pitch types with tiny counts (often mislabels).

In [ ]:
needed = ["pitch_type", "release_speed", "release_spin_rate", "pfx_x", "pfx_z", "description", "zone"]
print("Missing values:", pitches[needed].isna().sum().to_dict())
print("Pitches by type:", pitches["pitch_type"].value_counts(dropna=False).to_dict())

## 3. Arsenal: what each pitch is

In [ ]:
arsenal(pitches, min_n=MIN_N)

## 4. The same table in DuckDB SQL
Should match the pandas table. Rounding can differ by 0.1 because pandas rounds a trailing 5 to the even digit.

In [ ]:
print(ARSENAL_SQL)
arsenal_sql(pitches, min_n=MIN_N)

## 5. Outcomes: what happens when he throws it

In [ ]:
outcomes(pitches, min_n=MIN_N)

## 6. Usage against left- and right-handed hitters

In [ ]:
usage_by_hand(pitches, min_n=MIN_N)

## 7. Movement

In [ ]:
fig = movement_plot(pitches, label, f"Pitch movement, pitcher's view · {len(pitches):,} pitches · Statcast via Baseball Savant", min_n=MIN_N)